# Neo4j With py-dockerdb

Graph databases excel at questions that require traversing relationships — *who knows whom*, *which documents cite which papers*, *what entities co-occur across sources*. Dense-vector search cannot follow multi-hop connections; Neo4j can.

This notebook shows how `py-dockerdb` provisions a local Neo4j 5 instance in one call, then demonstrates basic Cypher operations and the connection handoff to LlamaIndex and LangChain for GraphRAG pipelines.

## Table Of Contents

- [Prerequisites](#prerequisites)
- [1. Start Neo4j](#1-start-neo4j)
- [2. Write And Read Nodes](#2-write-and-read-nodes)
- [3. Create And Traverse Relationships](#3-create-and-traverse-relationships)
- [4. LlamaIndex Integration (GraphRAG)](#4-llamaindex-integration-graphrag)
- [5. LangChain Integration](#5-langchain-integration)
- [6. Cleanup](#6-cleanup)

## Prerequisites

- Docker Desktop must be running before `create_db()` is called.
- Install dependencies:
  ```bash
  pip install -e ".[graph]"
  # or individually:
  pip install neo4j llama-index-graph-stores-neo4j langchain-community
  ```
- No environment variables required — credentials are set in the config below.

## 1. Start Neo4j

In [ ]:
import sys
import uuid
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

from docker_db.dbs.neo4j_db import Neo4jConfig, Neo4jDB

In [ ]:
temp_dir = Path("tmp")
temp_dir.mkdir(exist_ok=True)

container_name = f"demo-neo4j-{uuid.uuid4().hex[:8]}"

config = Neo4jConfig(
    password="demopassword",
    project_name="demo",
    container_name=container_name,
    workdir=temp_dir.absolute(),
    retries=20,
    delay=3,
)

db_manager = Neo4jDB(config)
db_manager.create_db()

print(f"Neo4j started:    {container_name}")
print(f"Bolt URI:         {db_manager.connection_string()}")
print(f"Browser UI:       http://localhost:{config.http_port}")
print(f"Credentials:      {config.user} / {config.password}")

## 2. Write And Read Nodes

`db_manager.connection` returns a `neo4j.Driver`. Open a session and run Cypher directly.

In [ ]:
driver = db_manager.connection

with driver.session(database=config.database) as session:
    session.run(
        "CREATE (p:Person {name: $name, role: $role})",
        name="Alice", role="researcher",
    )
    session.run(
        "CREATE (p:Person {name: $name, role: $role})",
        name="Bob", role="engineer",
    )
    result = session.run("MATCH (p:Person) RETURN p.name AS name, p.role AS role")
    for record in result:
        print(record["name"], "→", record["role"])

## 3. Create And Traverse Relationships

Relationships are first-class citizens in Neo4j. Multi-hop traversal is a single Cypher pattern.

In [ ]:
with driver.session(database=config.database) as session:
    # Create a KNOWS relationship
    session.run(
        """
        MATCH (a:Person {name: 'Alice'}), (b:Person {name: 'Bob'})
        CREATE (a)-[:KNOWS {since: 2022}]->(b)
        """
    )

    # Add a third node and chain the relationship
    session.run(
        "CREATE (p:Person {name: 'Carol', role: 'designer'})"
    )
    session.run(
        """
        MATCH (b:Person {name: 'Bob'}), (c:Person {name: 'Carol'})
        CREATE (b)-[:KNOWS {since: 2023}]->(c)
        """
    )

    # Two-hop traversal: who does Alice know through Bob?
    result = session.run(
        """
        MATCH (a:Person {name: 'Alice'})-[:KNOWS*2]->(c)
        RETURN c.name AS name, c.role AS role
        """
    )
    print("Alice knows (2 hops):")
    for record in result:
        print(" ", record["name"], "→", record["role"])

## 4. LlamaIndex Integration (GraphRAG)

`Neo4jGraphStore` uses the same Bolt connection to store and query the knowledge graph that LlamaIndex builds from documents.
Swap this for your actual documents and an embedding model for a full GraphRAG pipeline.

In [ ]:
try:
    from llama_index.graph_stores.neo4j import Neo4jGraphStore
    from llama_index.core import StorageContext, KnowledgeGraphIndex
    from llama_index.core.schema import Document

    graph_store = Neo4jGraphStore(
        username=config.user,
        password=config.password,
        url=db_manager.connection_string(),
        database=config.database,
    )

    storage_context = StorageContext.from_defaults(graph_store=graph_store)

    documents = [
        Document(text="Alice is a researcher who studies graph databases."),
        Document(text="Bob is an engineer who works with Alice on RAG systems."),
    ]

    index = KnowledgeGraphIndex.from_documents(
        documents,
        storage_context=storage_context,
        max_triplets_per_chunk=3,
        include_embeddings=False,
    )

    query_engine = index.as_query_engine(
        include_text=True,
        retriever_mode="keyword",
    )

    response = query_engine.query("What does Alice study?")
    print("LlamaIndex response:", response)

except ImportError:
    print("llama-index-graph-stores-neo4j not installed — skipping.")
    print("Install with: pip install 'py-dockerdb[graph]'")

## 5. LangChain Integration

In [ ]:
try:
    from langchain_community.graphs import Neo4jGraph

    graph = Neo4jGraph(
        url=db_manager.connection_string(),
        username=config.user,
        password=config.password,
        database=config.database,
    )

    result = graph.query("MATCH (p:Person) RETURN p.name AS name LIMIT 5")
    print("LangChain Neo4jGraph query result:", result)

except ImportError:
    print("langchain-community not installed — skipping.")
    print("Install with: pip install 'py-dockerdb[graph]'")

## 6. Cleanup

In [ ]:
driver.close()
db_manager.delete_db(running_ok=True)
print(f"Neo4j container '{container_name}' deleted.")